# ДЗ 2 (Пащенко Дмитрий)

Всё локально у меня в `hw2/`. ВМ нету... vm1 - мой ноут, подключение локальное.

На 5432 у меня уже крутится постгра под другой проект, поэтому мой на **5434**, служба postgres-custom.

In [1]:
%%bash
cat /etc/os-release | head -2
ansible --version | head -1
# этот не трогаю
systemctl is-active postgresql.service
ss -tln | grep 5432

NAME="Arch Linux"
PRETTY_NAME="Arch Linux"
ansible [core 2.21.5]
active
LISTEN 0      0               127.0.0.1:5432       0.0.0.0:*
LISTEN 0      0                   [::1]:5432             *:*


In [2]:
%%bash
cd hw2
find . -path ./.git -prune -o -type f -print | sort

./ansible.cfg
./.gitignore
./hw2_Пащенко_Дмитрий.ipynb
./inventories/main/group_vars/a

ll/main.yml
./inventories/main/group_vars/all/vault.yml
./inventories/main/hosts.yaml
./inventories/

main/host_vars/vm1.yml
./postgres.play.yml
./README.md
./requirements.yml
./roles/postgres/defaults/

main.yml
./roles/postgres/handlers/main.yml
./roles/postgres/README.md
./roles/postgres/tasks/main.y

ml
./roles/postgres/templates/pg_hba.conf.j2
./roles/postgres/templates/postgres-exporter.service.j2


./roles/postgres/templates/postgres.service.j2
./roles/postgres/vars/main.yml
./.vault_pass


## Подготовка

In [3]:
%%bash
cd hw2
ansible-galaxy collection install -r requirements.yml 2>&1

Starting galaxy collection install process
Nothing to do. All requested collections are already installed. If you want to reinstall them, consider using `--force`.


пасы рандомные и сразу шиффрую. `.vault_pass` в `.gitignore`.

In [4]:
%%bash
cd hw2
openssl rand -hex 24 > .vault_pass
chmod 600 .vault_pass

cat > inventories/main/group_vars/all/vault.yml <<EOF
vault_postgres_app_password: "$(openssl rand -hex 16)"
vault_postgres_analyst_password: "$(openssl rand -hex 16)"
vault_postgres_exporter_password: "$(openssl rand -hex 16)"
EOF
ansible-vault encrypt inventories/main/group_vars/all/vault.yml
cat .gitignore

.vault_pass
*.retry


In [5]:
%%bash
cd hw2
# так лежит в репе
cat inventories/main/group_vars/all/vault.yml
echo
# расшифровка, значения замазал
ansible-vault view inventories/main/group_vars/all/vault.yml | sed -E 's/: .*/: "***"/'

$ANSIBLE_VAULT;1.1;AES256
33666531353461636237386538306465396633663966366532303266343865303033353161363732
6130646536663263386237356131316266306236336161660a346434636463333531666566626330
38363563366535393435636462393261646365373636313935336663326533333830396238313563
3630613139396538320a663437323335396266303036643930333036336664616431363031323064
38613661363634316361373362643565363438336461623933636562613066316463613134376633
31363931383566383836396666643564373865326366343536343930633363363264653163333234
34376434353835306636313965373633326433623232313237643734353637376230613864626164
32653232303434333162643037666663323633373434366365653266656164633165643661646435
39383131663166623864333266396639336364663566613431663731643864613731353036616264
66336461353638363764303134373434653234393831316262393435303161363631656232303035
32386365666263313265333430303262303961353438306430313663383336643134366563613866
65313338326632303135373061643432356461643961653038646365636364333266336537646431
39

In [6]:
%%bash
cd hw2
ansible vm1 -m ping
ansible-playbook postgres.play.yml --syntax-check

vm1 | SUCCESS => {
    "changed": false,
    "ping": "pong"
}

playbook: postgres.play.yml


## Первый прогон

In [7]:
%%bash
cd hw2
ansible-playbook postgres.play.yml 2>&1


PLAY [Install PostgreSQL on vm1] ***********************************************

TASK [Gathering Facts] *********************************************************
Tuesday 13 October 2026  00:13:12 +0300 (0:00:00.019)       0:00:00.019 ******* 
ok: [vm1]

TASK [postgres : Install PostgreSQL packages] **********************************
Tuesday 13 October 2026  00:13:13 +0300 (0:00:00.772)       0:00:00.791 ******* 
changed: [vm1]

TASK [postgres : Create /data] *************************************************
Tuesday 13 October 2026  00:13:14 +0300 (0:00:01.359)       0:00:02.150 ******* 
changed: [vm1]

TASK [postgres : Create data directories] **************************************
Tuesday 13 October 2026  00:13:15 +0300 (0:00:00.268)       0:00:02.419 ******* 
changed: [vm1] => (item=/data/postgres)
changed: [vm1] => (item=/data/postgres/data)

TASK [postgres : Init database cluster] ****************************************
Tuesday 13 October 2026  00:13:15 +0300 (0:00:00.430)      

## Второй прогон, должно быть changed=0

In [8]:
%%bash
cd hw2
ansible-playbook postgres.play.yml 2>&1


PLAY [Install PostgreSQL on vm1] ***********************************************

TASK [Gathering Facts] *********************************************************
Tuesday 13 October 2026  00:13:12 +0300 (0:00:00.019)       0:00:00.019 ******* 
ok: [vm1]

TASK [postgres : Install PostgreSQL packages] **********************************
Tuesday 13 October 2026  00:13:13 +0300 (0:00:00.772)       0:00:00.791 ******* 
changed: [vm1]

TASK [postgres : Create /data] *************************************************
Tuesday 13 October 2026  00:13:14 +0300 (0:00:01.359)       0:00:02.150 ******* 
changed: [vm1]

TASK [postgres : Create data directories] **************************************
Tuesday 13 October 2026  00:13:15 +0300 (0:00:00.268)       0:00:02.419 ******* 
changed: [vm1] => (item=/data/postgres)
changed: [vm1] => (item=/data/postgres/data)

TASK [postgres : Init database cluster] ****************************************
Tuesday 13 October 2026  00:13:15 +0300 (0:00:00.430)      

## Проверки из "Как проверить себя"
У psql указываю сокет и порт, иначе уйдёт в ту постгру на 5432

In [9]:
%%bash
cd hw2
ansible vm1 -b -m command -a 'systemctl is-active postgres-custom postgres-exporter'
ansible vm1 -b -m command -a 'systemctl is-enabled postgres-custom postgres-exporter'

vm1 | CHANGED | rc=0 >>
active
active
vm1 | CHANGED | rc=0 >>
enabled
enabled


In [10]:
%%bash
cd hw2
ansible vm1 -b --become-user postgres -m command -a 'psql -h /run/postgres-custom -p 5434 -c "SELECT 1"'

vm1 | CHANGED | rc=0 >>
 ?column? 
----------
        1
(1 row)


In [11]:
%%bash
cd hw2
ansible vm1 -b --become-user postgres -m command -a 'psql -h /run/postgres-custom -p 5434 -c "\l"'

vm1 | CHANGED | rc=0 >>
                                                 List of databases
   Name    |  Owner   | Encoding | Locale Provider | Collate |  Ctype  | Locale | ICU Rules |   Access privileges   
-----------+----------+----------+-----------------+---------+---------+--------+-----------+-----------------------
 appdb     | app      | UTF8     | libc            | C.UTF-8 | C.UTF-8 |        |           | 
 postgres  | postgres | UTF8     | libc            | C.UTF-8 | C.UTF-8 |        |           | 
 reports   | analyst  | UTF8     | libc            | C.UTF-8 | C.UTF-8 |        |           | 
 template0 | postgres | UTF8     | libc            | C.UTF-8 | C.UTF-8 |        |           | =c/postgres          +
           |          |          |                 |         |         |        |           | postgres=CTc/postgres
 template1 | postgres | UTF8     | libc            | C.UTF-8 | C.UTF-8 |        |           | =c/postgres          +
           |          |          |      

In [12]:
%%bash
cd hw2
ansible vm1 -b --become-user postgres -m command -a 'psql -h /run/postgres-custom -p 5434 -c "\du"'
# роли (pg_monitor) теперь видно через \drg
ansible vm1 -b --become-user postgres -m command -a 'psql -h /run/postgres-custom -p 5434 -c "\drg"'

vm1 | CHANGED | rc=0 >>
                                 List of roles
     Role name     |                         Attributes                         
-------------------+------------------------------------------------------------
 analyst           | Create DB
 app               | 
 postgres          | Superuser, Create role, Create DB, Replication, Bypass RLS
 postgres_exporter | 
vm1 | CHANGED | rc=0 >>
                   List of role grants
     Role name     | Member of  |   Options    | Grantor  
-------------------+------------+--------------+----------
 postgres_exporter | pg_monitor | INHERIT, SET | postgres
(1 row)


In [13]:
%%bash
cd hw2
ansible vm1 -m uri -a 'url=http://127.0.0.1:9187/metrics return_content=true' | grep -o 'pg_up 1'

pg_up 1


## Каталоги и конфиг

In [14]:
%%bash
ls -ld /data /data/postgres
sudo ls -ld /data/postgres/data
sudo cat /data/postgres/data/PG_VERSION

drwxr-xr-x 3 root     root     4096 Oct 13 00:13 /data
drwx------ 3 postgres postgres 4096 Oct 13 00:13 /data/postgres
drwx------ 19 postgres postgres 4096 Oct 13 00:13 /data/postgres/data
18


In [15]:
%%bash
# поменялись только мои параметры
sudo grep -nE "^(port|unix_socket_directories|listen_addresses|max_connections|shared_buffers|password_encryption|log_min_duration_statement|log_line_prefix) " /data/postgres/data/postgresql.conf
echo "всего незакомментированных строк: $(sudo grep -cE '^[a-z]' /data/postgres/data/postgresql.conf)"

60:listen_addresses = 'localhost'
64:port = 5434
65:max_connections = 50
68:unix_socket_directories = '/run/postgres-custom'
97:password_encryption = 'scram-sha-256'
132:shared_buffers = '256MB'
561:log_min_duration_statement = 500
605:log_line_prefix = '%m [%p] %u@%d '
всего незакомментированных строк: 20


## Хендлеры
Меняю max_connections (нужен рестарт) и добавляю правило в pg_hba с релоадом

In [16]:
%%bash
cd hw2
sudo -u postgres psql -h /run/postgres-custom -p 5434 -tAc "SHOW max_connections"
sed -i 's/max_connections: 50/max_connections: 60/' inventories/main/host_vars/vm1.yml
cat >> inventories/main/host_vars/vm1.yml <<'EOF'

# правила pg_hba: дефолтные + доступ для analyst из локальной docker-сети
postgres__hba:
  - { type: local, database: all, user: postgres, method: peer }
  - { type: local, database: all, user: all, method: peer }
  - { type: host, database: all, user: all, address: 127.0.0.1/32, method: scram-sha-256 }
  - { type: host, database: all, user: all, address: "::1/128", method: scram-sha-256 }
  - { type: host, database: reports, user: analyst, address: 172.17.0.0/16, method: scram-sha-256 }
EOF
cat inventories/main/host_vars/vm1.yml

50
# для vm1 чуть поменяю конфиг относительно дефолтов роли (проверка что словарь мерджится)
postgres__opts:
  listen_addresses: localhost
  max_connections: 60
  shared_buffers: 256MB
  password_encryption: scram-sha-256
  log_min_duration_statement: 500
  log_line_prefix: "%m [%p] %u@%d "

# правила pg_hba: дефолтные + доступ для analyst из локальной docker-сети
postgres__hba:
  - { type: local, database: all, user: postgres, method: peer }
  - { type: local, database: all, user: all, method: peer }
  - { type: host, database: all, user: all, address: 127.0.0.1/32, method: scram-sha-256 }
  - { type: host, database: all, user: all, address: "::1/128", method: scram-sha-256 }
  - { type: host, database: reports, user: analyst, address: 172.17.0.0/16, method: scram-sha-256 }


In [17]:
%%bash
cd hw2
ansible-playbook postgres.play.yml 2>&1 | grep -E -A2 'TASK \[postgres : Configure|HANDLER|PLAY RECAP' | grep -vE '^(--|Tuesday|Monday|Wednesday|Thursday|Friday|Saturday|Sunday)'

TASK [postgres : Configure postgresql.conf] ************************************
ok: [vm1] => (item=port)
TASK [postgres : Configure pg_hba.conf] ****************************************
changed: [vm1]
RUNNING HANDLER [postgres : Restart PostgreSQL] ********************************
changed: [vm1]
RUNNING HANDLER [postgres : Reload PostgreSQL] *********************************
changed: [vm1]
PLAY RECAP *********************************************************************
vm1                        : ok=23   changed=4    unreachable=0    failed=0    skipped=1    rescued=0    ignored=0   



In [18]:
%%bash
sudo -u postgres psql -h /run/postgres-custom -p 5434 -tAc "SHOW max_connections"
sudo tail -2 /data/postgres/data/pg_hba.conf
# видно рестарт и reload
sudo journalctl -u postgres-custom --since "-2min" --no-pager | grep -E 'received SIGHUP|shutting down|ready to accept|Stopped|Started|Reloaded'

60
host    all             all             ::1/128                 scram-sha-256
host    reports         analyst         172.17.0.0/16           scram-sha-256
Oct 13 00:13:08 pd postgres[869200]: 2026-10-13 00:13:08.703 EEST [869200] @ LOG:  shutting down
Oct 13 00:13:08 pd systemd[1]: Stopped PostgreSQL database server (/data/postgres/data).
Oct 13 00:13:19 pd postgres[876259]: 2026-10-13 00:13:19.652 EEST [876259] @ LOG:  database system is ready to accept connections
Oct 13 00:13:19 pd systemd[1]: Started PostgreSQL database server (/data/postgres/data).
Oct 13 00:13:19 pd postgres[876259]: 2026-10-13 00:13:19.988 EEST [876259] @ LOG:  received SIGHUP, reloading configuration files
Oct 13 00:13:19 pd systemd[1]: Reloaded PostgreSQL database server (/data/postgres/data).
Oct 13 00:13:52 pd postgres[876264]: 2026-10-13 00:13:52.493 EEST [876264] @ LOG:  shutting down
Oct 13 00:13:52 pd systemd[1]: Stopped PostgreSQL database server (/data/postgres/data).
Oct 13 00:13:52 pd postgres[87

In [19]:
%%bash
cd hw2
# и снова changed=0
ansible-playbook postgres.play.yml 2>&1 | grep -A2 'PLAY RECAP'

PLAY RECAP *********************************************************************
vm1                        : ok=21   changed=0    unreachable=0    failed=0    skipped=1    rescued=0    ignored=0   

